# Tabla de Contenidos
1. [Preparación y exploración del conjunto de datos](#Preparación-y-exploración-del-conjunto-de-datos)
2. [Subcategoría Knife](#Subcategoría-Knife)
    1. [Division del dataset](#Division-del-dataset)
    2. [Entrenamiento](#Entrenamiento)
    3. [Validación con nuevas imágenes](#Validación-con-nuevas-imágenes)
    4. [Entrenamiento con augmentation](#Entrenamiento-con-augmentation)
    5. [Validación (con augmentation)](#Validación-externa-del-modelo-multiclase-con-imágenes-no-vistas-con-augmentation)
3. [Subcategorías knife + pistol](#Subcategorías-knife-pistol)
    1. [Incorporación y partición del conjunto de datos de pistolas en el dataset multiclase](#Incorporación-y-partición-del-conjunto-de-datos-de-pistolas-en-el-dataset-multiclase)
    2. [Entrenamiento del modelo multiclase (versión 1)](#Entrenamiento-versión-1)
    3. [Validación externa del modelo multiclase con imágenes no vistas](#Validación-externa-del-modelo-multiclase-con-imágenes-no-vistac)
    4. [Entrenamiento del modelo multiclase con augmentation (versión 2)](#Entrenamiento-del-modelo-multiclase-con-augmentation-versión-2)
    5. [Validación externa del modelo multiclase con imágenes no vistas (con augmentation)](#Validación-externa-del-modelo-multiclase-con-imágenes-no-vistas-con-augmentation-1)
    6. [Entrenamiento del modelo multiclase con augmentation (versión 3)](#Entrenamiento-del-modelo-multiclase-con-augmentation-versión-2)

    7. [Validación externa del modelo multiclase con imágenes no vistas](#Entrenamiento-del-modelo-multiclase-con-augmentation-versión-3)
    8. [Entrenamiento del modelo multiclase con augmentation (versión 4)](#Entrenamiento-del-modelo-multiclase-con-augmentation-versión-2)
    9. [Validación externa del modelo multiclase con imágenes no vistas](#Validación-externa-del-modelo-multiclase-con-imágenes-no-vistas-con-augmentation-1)
4. [Prueba con YOLOv8m](#Prueba-con-YOLOv8m)
    1. [Validación con DatasetVal2](#validar-con-datasetval2)


##  Preparación y exploración del conjunto de datos

In [ ]:
# Se instala YOLO.

!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
# Se carga el modelo.

model = YOLO('yolov8n.pt')  # versión pequeña (rápida)

In [ ]:
# Se Descarga el dataset.

!git clone https://github.com/ari-dasci/OD-WeaponDetection.git

In [ ]:
#  Identificación de categorías del conjunto de datos.
!ls "/content/OD-WeaponDetection/Weapons and similar handled objects"

In [ ]:
# Identificación de la estructura del conjunto de datos.
!find "/content/OD-WeaponDetection/Weapons and similar handled objects" -maxdepth 2 -type d

In [ ]:
# cuántas imágenes tiene cada subcategoría.

!find "/content/OD-WeaponDetection/Weapons and similar handled objects/Sohas_weapon-Classification/knife" | wc -l

!find "/content/OD-WeaponDetection/Weapons and similar handled objects/Sohas_weapon-Classification/pistol" | wc -l

!find "/content/OD-WeaponDetection/Weapons and similar handled objects/Sohas_weapon-Classification/smartphone" | wc -l

!find "/content/OD-WeaponDetection/Weapons and similar handled objects/Sohas_weapon-Classification/monedero" | wc -l

!find "/content/OD-WeaponDetection/Weapons and similar handled objects/Sohas_weapon-Classification/tarjeta" | wc -l

!find "/content/OD-WeaponDetection/Weapons and similar handled objects/Sohas_weapon-Classification/billete" | wc -l

In [ ]:
# Esto confirma que el dataset se cargó bien.

base_path = "/content/OD-WeaponDetection/Knife_detection" #guarda la ruta del dataset
import os

os.listdir(base_path) #lista archivos dentro de esa carpeta

In [ ]:
# Muestra los primeros archivos de la carpeta annotations.

labels_path = base_path + "/annotations"
os.listdir(labels_path)[:5]

In [ ]:
# Se crea el dataframe.

import pandas as pd
import xml.etree.ElementTree as ET
import os

data = []

labels_path = "/content/OD-WeaponDetection/Knife_detection/annotations"

for file in os.listdir(labels_path):
    if file.endswith(".xml"):
        tree = ET.parse(os.path.join(labels_path, file))
        root = tree.getroot()

        for obj in root.findall("object"):
            name = obj.find("name").text
            bbox = obj.find("bndbox")

            data.append({
                "imagen": file.replace(".xml", ".jpg"),
                "clase": name,
                "xmin": int(bbox.find("xmin").text),
                "ymin": int(bbox.find("ymin").text),
                "xmax": int(bbox.find("xmax").text),
                "ymax": int(bbox.find("ymax").text)
            })

df = pd.DataFrame(data)
df.head()

In [ ]:
df['clase'].value_counts()

In [ ]:
df.describe()

In [ ]:
# Esto lee el XML, extrae el boundig box, calcula las coordenadas Yolo y crea el archivo txt.

import xml.etree.ElementTree as ET
from PIL import Image

#  RUTAS

base_path = "/content/OD-WeaponDetection/Knife_detection"
images_path = base_path + "/Images"
labels_xml_path = base_path + "/annotations"

# Carpeta salida YOLO
yolo_labels_path = base_path + "/labels"
os.makedirs(yolo_labels_path, exist_ok=True)


for file in os.listdir(labels_xml_path):

    if file.endswith(".xml"):

        tree = ET.parse(os.path.join(labels_xml_path, file)) # Leer archivo XML
        root = tree.getroot()

        # nombre base sin extensión
        image_name = file.replace(".xml", "")

        # buscar imagen real (.jpg o .png)
        if os.path.exists(os.path.join(images_path, image_name + ".jpg")):
            image_path = os.path.join(images_path, image_name + ".jpg")
        elif os.path.exists(os.path.join(images_path, image_name + ".png")):
            image_path = os.path.join(images_path, image_name + ".png")
        else:
            print(f"⚠️ Imagen no encontrada: {image_name}")
            continue

        # abrir imagen
        img = Image.open(image_path)# Obtener tamaño de la imagen
        width, height = img.size

        # archivo de salida YOLO
        yolo_file_path = os.path.join(yolo_labels_path, file.replace(".xml", ".txt"))
        yolo_file = open(yolo_file_path, "w")

        # recorrer objetos
        for obj in root.findall("object"):

            cls = 0  # knife = clase 0

            bbox = obj.find("bndbox")
            xmin = int(bbox.find("xmin").text)
            ymin = int(bbox.find("ymin").text)
            xmax = int(bbox.find("xmax").text)
            ymax = int(bbox.find("ymax").text)



            # Convertir coordenadas a formato YOLO

            x_center = ((xmin + xmax) / 2) / width
            y_center = ((ymin + ymax) / 2) / height
            w = (xmax - xmin) / width
            h = (ymax - ymin) / height

            # escribir en archivo
            yolo_file.write(f"{cls} {x_center} {y_center} {w} {h}\n")

        yolo_file.close()

print("✅ Conversión a formato YOLO completada")

In [ ]:
os.listdir("/content/OD-WeaponDetection/Knife_detection")

In [ ]:
len(os.listdir("/content/OD-WeaponDetection/Knife_detection/labels"))

## Subcategoría Knife

### Division del dataset





In [ ]:
# Dividir dataset (train / val)

import shutil
import random

base_path = "/content/OD-WeaponDetection/Knife_detection"

images_path = base_path + "/Images"
labels_path = base_path + "/labels"

# Crear carpetas
for split in ["train", "val"]:
    os.makedirs(f"{base_path}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_path}/labels/{split}", exist_ok=True)

# Lista de archivos (sin extensión)
files = [f.replace(".txt", "") for f in os.listdir(labels_path)]

random.shuffle(files)

# División 80/20
split_index = int(0.8 * len(files))
train_files = files[:split_index]
val_files = files[split_index:]

def copiar(files, split):
    for name in files:
        for ext in [".jpg", ".png"]:
            img_path = os.path.join(images_path, name + ext)
            if os.path.exists(img_path):
                shutil.copy(img_path, f"{base_path}/images/{split}/{name + ext}")
                shutil.copy(f"{labels_path}/{name}.txt", f"{base_path}/labels/{split}/{name}.txt")
                break

copiar(train_files, "train")
copiar(val_files, "val")

print("Dataset dividido correctamente")


In [ ]:
# Verificación de la estructura.
# Lista los archivos dentro de una carpeta y solo muestra los primeros 5 archivos.

os.listdir(base_path + "/images/train")[:5]


In [ ]:
# Los .txt también están.
os.listdir(base_path + "/labels/train")[:5]

In [ ]:
# Crear archivo data.yaml
# Aquí se especifica la ubicación de los datos de entrenamiento y validación, así como las clases del problema, permitiendo que el modelo YOLOv8 pueda acceder correctamente a la información durante el proceso de entrenamiento.
data_yaml = f"""
path: {base_path}
train: images/train
val: images/val

names:
  0: knife
"""

with open(base_path + "/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml creado")

### Entrenamiento

In [ ]:
# Entrenamiento de YOLO.
model.train(
    data=base_path + "/data.yaml",
    epochs=50,
    imgsz=640
)

In [ ]:
# Resultados
results = model.predict(
    source="/content/OD-WeaponDetection/Knife_detection/images/val",
    save=True
)

In [ ]:
# Muestra imágenes detectadas.
import matplotlib.pyplot as plt
import cv2
import os

predict_path = "/content/runs/detect/predict"

images = os.listdir(predict_path)

# mostrar 3 imágenes
for img_name in images[:3]:
    img = cv2.imread(os.path.join(predict_path, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img)
    plt.title(img_name)
    plt.axis("off")
    plt.show()

In [ ]:
# Guarda el modelo.
from google.colab import files
files.download('/content/runs/detect/train/weights/best.pt')

In [ ]:
# Mostrar métricas finales.
# El modelo genera estructuras de datos que contienen las predicciones, incluyendo las coordenadas de los objetos detectados, las clases y los tiempos de inferencia.
print(results)

### Validación con nuevas imágenes

Se considera 10 imágenes nuevas, que no están en el dataset utilizado.




In [ ]:
!unzip /content/DatasetValidacion.zip

In [ ]:
!ls /content/DatasetValidacion

In [ ]:
# Cargar modelo entrenado
model = YOLO("/content/runs/detect/train/weights/best.pt")

# Probar imágenes externas
results = model.predict(
    source="/content/DatasetValidacion",
    save=True,
    conf=0.5
)

In [ ]:
import os
from IPython.display import Image, display

# Ruta de resultados
results_path = "/content/runs/detect/predict-2"

# Mostrar imágenes
for img_name in os.listdir(results_path):
    img_path = os.path.join(results_path, img_name)
    display(Image(filename=img_path))

### Entrenamiento con augmentation

In [ ]:
# Nuevo entrenamiento con augmentation

model.train(
    data="/content/OD-WeaponDetection/Knife_detection/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="train_aug",

    degrees=20,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_v=0.4
)

### Validación

In [ ]:
from ultralytics import YOLO

# Cargar modelo con augmentation
model = YOLO("/content/runs/detect/train_aug/weights/best.pt")

# Probar imágenes externas
results = model.predict(
    source="/content/DatasetValidacion",
    save=True,
    conf=0.5
)

In [ ]:
import os
from IPython.display import Image, display

results_path = "/content/runs/detect/predict-4"

for img_name in os.listdir(results_path):
    img_path = os.path.join(results_path, img_name)
    display(Image(filename=img_path))

## Subcategorías knife + pistol

In [ ]:
# Exploración de la estructura general del repositorio OD-WeaponDetection.
!ls /content/OD-WeaponDetection

In [ ]:
# Verificación de la estructura del dataset de detección de pistolas.
!ls /content/OD-WeaponDetection/"Pistol detection"

In [ ]:
import xml.etree.ElementTree as ET

# Rutas
xml_dir = "/content/OD-WeaponDetection/Pistol detection/xmls"
img_dir = "/content/OD-WeaponDetection/Pistol detection/Weapons"

# Carpeta labels YOLO
labels_dir = "/content/OD-WeaponDetection/Pistol detection/labels"
os.makedirs(labels_dir, exist_ok=True)

# Clase pistol = 1
class_id = 1

# Convertir XML a YOLO
for xml_file in os.listdir(xml_dir):

    if not xml_file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(xml_dir, xml_file))
    root = tree.getroot()

    # Tamaño imagen
    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)

    yolo_lines = []

    # Objetos
    for obj in root.findall("object"):

        bndbox = obj.find("bndbox")

        xmin = int(bndbox.find("xmin").text)
        ymin = int(bndbox.find("ymin").text)
        xmax = int(bndbox.find("xmax").text)
        ymax = int(bndbox.find("ymax").text)

        # Conversión a YOLO
        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height
        w = (xmax - xmin) / width
        h = (ymax - ymin) / height

        yolo_lines.append(
            f"{class_id} {x_center} {y_center} {w} {h}"
        )

    # Guarda txt
    txt_name = xml_file.replace(".xml", ".txt")

    with open(os.path.join(labels_dir, txt_name), "w") as f:
        f.write("\n".join(yolo_lines))

print("Conversión pistol a YOLO completada.")

In [ ]:
# Se crean carpetas multiclase.
base_multi = "/content/dataset_multiclase"

folders = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for folder in folders:
    os.makedirs(os.path.join(base_multi, folder), exist_ok=True)

print("Carpetas multiclase creadas.")

In [ ]:
# Copiar KNIFE al dataset multiclase.

import shutil
import random

# Knife paths
knife_img_train = "/content/OD-WeaponDetection/Knife_detection/images/train"
knife_img_val = "/content/OD-WeaponDetection/Knife_detection/images/val"

knife_lbl_train = "/content/OD-WeaponDetection/Knife_detection/labels/train"
knife_lbl_val = "/content/OD-WeaponDetection/Knife_detection/labels/val"

# Destino
multi_base = "/content/dataset_multiclase"

# Copiar train
for file in os.listdir(knife_img_train):
    shutil.copy(
        os.path.join(knife_img_train, file),
        os.path.join(multi_base, "images/train", file)
    )

for file in os.listdir(knife_lbl_train):
    shutil.copy(
        os.path.join(knife_lbl_train, file),
        os.path.join(multi_base, "labels/train", file)
    )

# Copiar val
for file in os.listdir(knife_img_val):
    shutil.copy(
        os.path.join(knife_img_val, file),
        os.path.join(multi_base, "images/val", file)
    )

for file in os.listdir(knife_lbl_val):
    shutil.copy(
        os.path.join(knife_lbl_val, file),
        os.path.join(multi_base, "labels/val", file)
    )

print(" Knife agregado al dataset multiclase")

### Incorporación y partición del conjunto de datos de pistolas en el dataset multiclase

In [ ]:
# Paths pistol
pistol_img_dir = "/content/OD-WeaponDetection/Pistol detection/Weapons"
pistol_lbl_dir = "/content/OD-WeaponDetection/Pistol detection/labels"

# Lista labels
files = [f.replace(".txt", "") for f in os.listdir(pistol_lbl_dir)]

random.shuffle(files)

split_index = int(0.8 * len(files))

train_files = files[:split_index]
val_files = files[split_index:]

# TRAIN
for name in train_files:

    img_file = name + ".jpg"
    lbl_file = name + ".txt"

    shutil.copy(
        os.path.join(pistol_img_dir, img_file),
        "/content/dataset_multiclase/images/train"
    )

    shutil.copy(
        os.path.join(pistol_lbl_dir, lbl_file),
        "/content/dataset_multiclase/labels/train"
    )

# VAL
for name in val_files:

    img_file = name + ".jpg"
    lbl_file = name + ".txt"

    shutil.copy(
        os.path.join(pistol_img_dir, img_file),
        "/content/dataset_multiclase/images/val"
    )

    shutil.copy(
        os.path.join(pistol_lbl_dir, lbl_file),
        "/content/dataset_multiclase/labels/val"
    )

print(" Pistol agregado correctamente")

In [ ]:
# Aquí se crea el nuevo data.yaml

%%writefile /content/dataset_multiclase/data.yaml

train: /content/dataset_multiclase/images/train
val: /content/dataset_multiclase/images/val

nc: 2

names:
  0: knife
  1: pistol

### Entrenamiento (versión 1)

In [ ]:
# Cargar modelo base
model = YOLO("yolov8n.pt")

# Entrenamiento multiclase
model.train(
    data="/content/dataset_multiclase/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="train_multiclase"
)

In [ ]:
!unzip -q /content/DatasetVal2.zip -d /content/DatasetVal2

In [ ]:
!ls /content/DatasetVal2/DatasetVal2

### Validación externa del modelo multiclase con imágenes no vista

In [ ]:
from ultralytics import YOLO

# Cargar modelo multiclase
model = YOLO("/content/runs/detect/train_multiclase/weights/best.pt")

# Validación externa
results = model.predict(
    source="/content/DatasetVal2/DatasetVal2",
    save=True,
    conf=0.5
)

In [ ]:
from IPython.display import Image, display
import os

pred_dir = "/content/runs/detect/predict"

for img in os.listdir(pred_dir):
    display(Image(os.path.join(pred_dir, img)))

### Entrenamiento del modelo multiclase con augmentation (versión 2)

In [ ]:

model = YOLO("yolov8n.pt")

model.train(
    data="/content/dataset_multiclase/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="train_multiclase_aug",

    degrees=20,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_v=0.4
)

### Validación externa del modelo multiclase con imágenes no vistas (con augmentation)





In [ ]:
model = YOLO("/content/runs/detect/train_multiclase_aug/weights/best.pt")

results = model.predict(
    source="/content/DatasetVal2/DatasetVal2",
    save=True,
    conf=0.5
)

In [ ]:

pred_dir = "/content/runs/detect/predict-2"   # carpeta que aparece en tu salida

for img in sorted(os.listdir(pred_dir)):
    print(img)
    display(Image(os.path.join(pred_dir, img)))

### Entrenamiento del modelo para augmentation (versión 3)

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="/content/dataset_multiclase/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="train_multiclase_aug_v3",

    degrees=15,
    translate=0.10,
    scale=0.3,
    fliplr=0.5,
    hsv_v=0.3,
    mosaic=0.5
)

### validacion

In [ ]:
# Cargar el nuevo modelo entrenado
model = YOLO("/content/runs/detect/train_multiclase_aug_v3/weights/best.pt")

# Validación externa
results = model.predict(
    source="/content/DatasetVal2/DatasetVal2",
    save=True,
    conf=0.5
)

In [ ]:
from IPython.display import Image, display
import os

folder = "/content/runs/detect/predict"

for img in sorted(os.listdir(folder)):
    display(Image(filename=os.path.join(folder, img), width=600))

### Entrenamiento del modelo con augmentation (versión 4)

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="/content/dataset_multiclase/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="train_multiclase_aug_v4",

    # Augmentation balanceado
    degrees=20,
    translate=0.20,
    scale=0.5,
    fliplr=0.5,
    hsv_v=0.4,
    mosaic=0.8
)

In [ ]:
# Cargar modelo v4
model = YOLO("/content/runs/detect/train_multiclase_aug_v4/weights/best.pt")

# Validación externa
results = model.predict(
    source="/content/DatasetVal2/DatasetVal2",
    save=True,
    conf=0.5
)

## Prueba con YOLOv8m

In [ ]:
model = YOLO("yolov8m.pt")

results = model.train(
    data="/content/dataset_multiclase/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/runs/detect",
    name="train_multiclase_yolov8m"
)

### validar con DatasetVal2

In [ ]:
model = YOLO("/content/runs/detect/train_multiclase_yolov8m/weights/best.pt")

results = model.predict(
    source="/content/DatasetVal2/DatasetVal2",
    save=True,
    conf=0.5
)